In [1]:
import requests
from pathlib import Path

PDB_ID = "7R73"

url = f"https://files.rcsb.org/download/{PDB_ID}.pdb"
pdb_text = requests.get(url).text

Path(f"{PDB_ID}.pdb").write_text(pdb_text)

print("Downloaded:", PDB_ID)
print("File size:", len(pdb_text), "characters")

Downloaded: 7R73
File size: 885978 characters


In [2]:
from collections import defaultdict

chains = defaultdict(set)

with open("7R73.pdb") as f:
    for line in f:
        if line.startswith("ATOM"):
            chain = line[21]
            resnum = line[22:26].strip()
            resname = line[17:20].strip()

            chains[chain].add((resnum, resname))

for chain, residues in chains.items():
    print(
        "Chain:", chain,
        "| residues:", len(residues)
    )

Chain: G | residues: 287
Chain: A | residues: 126


In [3]:
def extract_chain(input_pdb, output_pdb, chain_id):
    with open(input_pdb) as fin, open(output_pdb, "w") as fout:

        for line in fin:

            if line.startswith(("ATOM", "TER")):

                if len(line) > 21 and line[21] == chain_id:
                    fout.write(line)

        fout.write("END\n")


extract_chain(
    "7R73.pdb",
    "7R73_nanobody_experimental.pdb",
    "A"
)

extract_chain(
    "7R73.pdb",
    "7R73_antigen_gp120.pdb",
    "G"
)

print("Done")

Done


In [4]:
def count_atoms(filename):
    atoms = 0

    with open(filename) as f:
        for line in f:
            if line.startswith("ATOM"):
                atoms += 1

    return atoms


print(
    "Experimental nanobody atoms:",
    count_atoms("7R73_nanobody_experimental.pdb")
)

print(
    "Antigen atoms:",
    count_atoms("7R73_antigen_gp120.pdb")
)

Experimental nanobody atoms: 1877
Antigen atoms: 4466


In [5]:
import os

for f in [
    "7R73.pdb",
    "7R73_nanobody_experimental.pdb",
    "7R73_antigen_gp120.pdb"
]:
    print(f, os.path.getsize(f), "bytes")

7R73.pdb 885978 bytes
7R73_nanobody_experimental.pdb 152122 bytes
7R73_antigen_gp120.pdb 361831 bytes


In [6]:
import os

TARGET = "7R73A"

matches = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if TARGET.lower() in file.lower():
            matches.append(os.path.join(root, file))

print("Found:")

for m in matches:
    print(m)

Found:
/kaggle/input/notebooks/mennanabiil/docking/7R73A_NBB2.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73A_SimpleFold_DOCK.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73A.fasta
/kaggle/input/notebooks/mennanabiil/docking/7R73A_NBB2_DOCK.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73A_IgFold_DOCK.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73A_IgFold.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73A_SimpleFold.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A_NBB2.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A_SimpleFold_DOCK.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A.fasta
/kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A_NBB2_DOCK.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A_IgFold_DOCK.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A_IgFold.pdb
/kaggle/input/notebooks/mennanabiil/docking/7R73_d

In [7]:
import os
import json
import shutil

TARGET = "7R73A"

# --------------------------------------------------
# 1. Find targets_vhh.json and IgFold prediction
# --------------------------------------------------

targets_path = None
igfold_source = None

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        full = os.path.join(root, file)

        if file == "targets_vhh.json":
            targets_path = full

        if file.lower() == "7r73a_igfold.pdb":
            igfold_source = full

print("targets:", targets_path)
print("IgFold :", igfold_source)

assert targets_path is not None, "targets_vhh.json not found"
assert igfold_source is not None, "7R73A IgFold prediction not found"


# --------------------------------------------------
# 2. Get the 7R73A sequence
# --------------------------------------------------

with open(targets_path) as f:
    targets = json.load(f)

target = None

for x in targets:
    tid = x["pdb"] + x["H"]

    if tid.upper() == TARGET:
        target = x
        break

assert target is not None, f"{TARGET} not found"

sequence = target["hseq"]

print("\nTarget:", TARGET)
print("Sequence length:", len(sequence))
print("CDR-H3:", target["CDRH3"])
print("CDR-H3 length:", len(target["CDRH3"]))


# --------------------------------------------------
# 3. Save FASTA
# --------------------------------------------------

fasta_path = "/kaggle/working/7R73A.fasta"

with open(fasta_path, "w") as f:
    f.write(">7R73A\n")
    f.write(sequence + "\n")

print("\nSaved FASTA:", fasta_path)


# --------------------------------------------------
# 4. Copy existing IgFold prediction
# --------------------------------------------------

output_igfold = "/kaggle/working/7R73A_IgFold.pdb"

shutil.copy(
    igfold_source,
    output_igfold
)

print("Copied IgFold prediction:", output_igfold)


# --------------------------------------------------
# 5. Quick check
# --------------------------------------------------

def count_atoms(filename):
    with open(filename) as f:
        return sum(
            line.startswith("ATOM")
            for line in f
        )

print("\nIgFold atoms:", count_atoms(output_igfold))
print(
    "IgFold file size:",
    os.path.getsize(output_igfold),
    "bytes"
)

targets: None
IgFold : /kaggle/input/notebooks/mennanabiil/docking/7R73_docking_project/7R73A_IgFold.pdb


AssertionError: targets_vhh.json not found

In [ ]:
!apt-get update -qq
!apt-get install -y -qq hmmer

!pip install -q biopython openmm
!pip install -q git+https://github.com/openmm/pdbfixer.git

!rm -rf /kaggle/working/ANARCI
!git clone -q https://github.com/oxpig/ANARCI.git /kaggle/working/ANARCI

%cd /kaggle/working/ANARCI
!python setup.py install
%cd /kaggle/working

!pip install -q ImmuneBuilder

In [ ]:
import anarci
import pdbfixer
import openmm

from ImmuneBuilder import NanoBodyBuilder2

print("ANARCI ✓")
print("PDBFixer ✓")
print("OpenMM ✓", openmm.__version__)
print("NanoBodyBuilder2 ✓")

In [ ]:
from ImmuneBuilder import NanoBodyBuilder2
import os

predictor = NanoBodyBuilder2()

prediction = predictor.predict({
    "H": sequence
})

nbb2_output = "/kaggle/working/7R73A_NBB2.pdb"

prediction.save(nbb2_output)

print("Saved:", nbb2_output)
print("Atoms:", count_atoms(nbb2_output))
print("Size:", os.path.getsize(nbb2_output), "bytes")

In [ ]:
# Clean previous copy if there is one
!rm -rf /kaggle/working/ml-simplefold

# Same fork + branch used in our benchmark
!git clone -q \
    --branch fix-multichain-fasta-parsing \
    https://github.com/omar-A-hassan/ml-simplefold.git \
    /kaggle/working/ml-simplefold

# Install SimpleFold
%cd /kaggle/working/ml-simplefold
!pip install -q -e .

# Install ESM dependency
!pip install -q git+https://github.com/facebookresearch/esm.git

%cd /kaggle/working

print("Installation finished")

In [ ]:
import shutil

print("SimpleFold:", shutil.which("simplefold"))

In [ ]:
from pathlib import Path

inference_file = Path(
    "/kaggle/working/ml-simplefold/src/simplefold/inference.py"
)

text = inference_file.read_text()

old = "esm_model = esm_model.to(device)"
new = "esm_model = esm_model.half().to(device)"

if old in text:
    text = text.replace(old, new, 1)
    inference_file.write_text(text)
    print("ESM FP16 patch applied ✓")

elif new in text:
    print("ESM FP16 patch already applied ✓")

else:
    print("Could not find expected line")

In [ ]:
!grep -n "esm_model = esm_model" \
/kaggle/working/ml-simplefold/src/simplefold/inference.py

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
import shutil
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nSimpleFold:", shutil.which("simplefold"))

files_to_check = [
    "/kaggle/working/7R73A.fasta",
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_NBB2.pdb",
]

for f in files_to_check:
    print(os.path.basename(f), "✓" if os.path.exists(f) else "MISSING")

In [ ]:
from pathlib import Path

inference_file = Path(
    "/kaggle/working/ml-simplefold/src/simplefold/inference.py"
)

text = inference_file.read_text()

old = "esm_model = esm_model.to(device)"
new = "esm_model = esm_model.half().to(device)"

if new in text:
    print("Patch already applied ✓")
elif old in text:
    text = text.replace(old, new, 1)
    inference_file.write_text(text)
    print("Patch applied ✓")
else:
    print("Expected line not found")

In [ ]:
!grep -n "esm_model = esm_model" /kaggle/working/ml-simplefold/src/simplefold/inference.py

In [ ]:
!rm -rf /kaggle/working/simplefold_7R73A

!simplefold \
    --simplefold_model simplefold_100M \
    --num_steps 500 \
    --tau 0.01 \
    --nsample_per_protein 1 \
    --seed 12345 \
    --fasta_path /kaggle/working/7R73A.fasta \
    --output_dir /kaggle/working/simplefold_7R73A \
    --output_format pdb \
    --backend torch

In [ ]:
import os

found = []

for root, dirs, files in os.walk("/kaggle/working/simplefold_7R73A"):
    for file in files:
        if file.endswith(".pdb"):
            path = os.path.join(root, file)
            found.append(path)
            print(path)

print("\nTotal PDB files found:", len(found))

In [ ]:
import shutil
import os

simplefold_source = found[0]
simplefold_output = "/kaggle/working/7R73A_SimpleFold.pdb"

shutil.copy(simplefold_source, simplefold_output)

def count_atoms(filename):
    with open(filename) as f:
        return sum(line.startswith("ATOM") for line in f)

print("Saved:", simplefold_output)
print("Atoms:", count_atoms(simplefold_output))
print("Size:", os.path.getsize(simplefold_output), "bytes")

In [ ]:
import os

for f in [
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_NBB2.pdb",
    "/kaggle/working/7R73A_SimpleFold.pdb",
    "/kaggle/working/7R73_antigen_gp120.pdb",
    "/kaggle/working/7R73_nanobody_experimental.pdb",
]:
    print(os.path.basename(f), "✓" if os.path.exists(f) else "MISSING")

In [ ]:
import os

def pdb_summary(path):
    atoms = 0
    residues = set()
    chains = set()

    with open(path) as f:
        for line in f:
            if line.startswith("ATOM"):
                atoms += 1

                chain = line[21].strip() or "_"
                resnum = line[22:26].strip()
                icode = line[26].strip()

                chains.add(chain)
                residues.add((chain, resnum, icode))

    return {
        "file": os.path.basename(path),
        "atoms": atoms,
        "residues": len(residues),
        "chains": sorted(chains)
    }


files = [
    "/kaggle/working/7R73_antigen_gp120.pdb",
    "/kaggle/working/7R73_nanobody_experimental.pdb",
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_NBB2.pdb",
    "/kaggle/working/7R73A_SimpleFold.pdb",
]

for f in files:
    print(pdb_summary(f))

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import PDBFile
from pathlib import Path
import os


def strip_hydrogens_and_set_chain(input_pdb, output_pdb, new_chain):
    """
    Keep protein ATOM records only,
    remove hydrogen atoms,
    and assign one consistent chain ID.
    """
    
    with open(input_pdb) as fin, open(output_pdb, "w") as fout:
        for line in fin:

            if not line.startswith("ATOM"):
                continue

            atom_name = line[12:16].strip()
            element = line[76:78].strip().upper()

            # Detect hydrogen
            is_hydrogen = (
                element == "H"
                or (
                    not element
                    and atom_name.lstrip("0123456789").startswith("H")
                )
            )

            if is_hydrogen:
                continue

            # Change chain ID
            line = line[:21] + new_chain + line[22:]

            fout.write(line)

        fout.write("END\n")


def prepare_structure(input_pdb, output_pdb, chain_id):

    temp = "/kaggle/working/_temp_noH.pdb"

    strip_hydrogens_and_set_chain(
        input_pdb,
        temp,
        chain_id
    )

    fixer = PDBFixer(filename=temp)

    # Detect missing residues but DO NOT build them
    fixer.findMissingResidues()
    fixer.missingResidues = {}

    # Add only missing atoms in residues already present
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()

    with open(output_pdb, "w") as f:
        PDBFile.writeFile(
            fixer.topology,
            fixer.positions,
            f,
            keepIds=True
        )

    os.remove(temp)

    print("Prepared:", os.path.basename(output_pdb))

In [ ]:
prepare_structure(
    "/kaggle/working/7R73_antigen_gp120.pdb",
    "/kaggle/working/7R73_antigen_DOCK.pdb",
    "G"
)

prepare_structure(
    "/kaggle/working/7R73_nanobody_experimental.pdb",
    "/kaggle/working/7R73_experimental_DOCK.pdb",
    "H"
)

prepare_structure(
    "/kaggle/working/7R73A_IgFold.pdb",
    "/kaggle/working/7R73A_IgFold_DOCK.pdb",
    "H"
)

prepare_structure(
    "/kaggle/working/7R73A_NBB2.pdb",
    "/kaggle/working/7R73A_NBB2_DOCK.pdb",
    "H"
)

prepare_structure(
    "/kaggle/working/7R73A_SimpleFold.pdb",
    "/kaggle/working/7R73A_SimpleFold_DOCK.pdb",
    "H"
)

In [ ]:
dock_files = [
    "/kaggle/working/7R73_antigen_DOCK.pdb",
    "/kaggle/working/7R73_experimental_DOCK.pdb",
    "/kaggle/working/7R73A_IgFold_DOCK.pdb",
    "/kaggle/working/7R73A_NBB2_DOCK.pdb",
    "/kaggle/working/7R73A_SimpleFold_DOCK.pdb",
]

for f in dock_files:
    print(pdb_summary(f))

In [ ]:
import os
import shutil
from pathlib import Path

backup_dir = Path("/kaggle/working/7R73_docking_project")
backup_dir.mkdir(exist_ok=True)

important_files = [
    # Original complex
    "7R73.pdb",
    
    # Original separated structures
    "7R73_antigen_gp120.pdb",
    "7R73_nanobody_experimental.pdb",
    
    # Sequence
    "7R73A.fasta",
    
    # Raw model predictions
    "7R73A_IgFold.pdb",
    "7R73A_NBB2.pdb",
    "7R73A_SimpleFold.pdb",
    
    # Standardized docking inputs
    "7R73_antigen_DOCK.pdb",
    "7R73_experimental_DOCK.pdb",
    "7R73A_IgFold_DOCK.pdb",
    "7R73A_NBB2_DOCK.pdb",
    "7R73A_SimpleFold_DOCK.pdb",
]

for name in important_files:
    src = Path("/kaggle/working") / name
    
    if src.exists():
        shutil.copy2(src, backup_dir / name)
        print("✓", name)
    else:
        print("MISSING:", name)

# Create ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/7R73_docking_project_backup",
    "zip",
    backup_dir
)

print("\nBackup created:")
print(zip_path)
print("Size:", round(os.path.getsize(zip_path)/1024/1024, 2), "MB")

In [ ]:
import os

path = "/kaggle/working/7R73_docking_project_backup.zip"

print("Exists:", os.path.exists(path))
print("Size:", round(os.path.getsize(path)/1024/1024, 2), "MB")

In [ ]:
import os
import subprocess

for root, dirs, files in os.walk("/kaggle/input"):
    if "Top10Docking" in root or "top10" in root.lower():
        for name in files:
            path = os.path.join(root, name)
            print("\n", path)
            subprocess.run(["file", path])

In [ ]:
import tarfile
from pathlib import Path

base = Path("/kaggle/input/datasets/mennanabiil/top10docking")
out_base = Path("/kaggle/working/7R73_HDOCK_top10")

archives = {
    "EXP": base / "top10_models-exp",
    "IgFold": base / "top10_models-igfold",
    "NBB2": base / "top10_models-nano",
    "SimpleFold": base / "top10_models-simple",
}

for name, archive in archives.items():
    out_dir = out_base / name
    out_dir.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive, "r") as tar:
        tar.extractall(out_dir)

    pdbs = sorted(out_dir.rglob("*.pdb"))

    print(f"\n{name}: {len(pdbs)} PDB files")
    for p in pdbs:
        print(" ", p.name)

In [ ]:
import requests
from pathlib import Path

r = requests.get("https://files.rcsb.org/download/7R73.pdb")
r.raise_for_status()

path = Path("/kaggle/working/7R73.pdb")
path.write_bytes(r.content)

print("Saved:", path)
print("Size:", path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/7R73_HDOCK_top10")

for name in ["EXP", "IgFold", "NBB2", "SimpleFold"]:
    folder = base / name
    pdbs = sorted(folder.rglob("*.pdb"))

    print(f"\n{name}: {len(pdbs)} PDB files")

    for p in pdbs:
        print(" ", p)

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/7R73_HDOCK_top10")

def chain_summary(pdb):
    chains = {}

    with open(pdb) as f:
        for line in f:
            if line.startswith("ATOM"):
                chain = line[21].strip() or "_"
                residue = (line[22:26].strip(), line[26].strip())
                chains.setdefault(chain, set()).add(residue)

    return {c: len(r) for c, r in chains.items()}


print("Native 7R73:")
print(chain_summary("/kaggle/working/7R73.pdb"))

for name in ["EXP", "IgFold", "NBB2", "SimpleFold"]:

    model1 = next((base / name).rglob("model_1.pdb"))

    print(f"\n{name} model_1:")
    print(model1)
    print(chain_summary(model1))

In [ ]:
from pathlib import Path
from collections import Counter

base = Path("/kaggle/working/7R73_HDOCK_top10")

for name in ["EXP", "IgFold", "NBB2", "SimpleFold"]:
    pdb = next((base / name).rglob("model_1.pdb"))

    record_counts = Counter()
    chain_counts = Counter()

    atom_lines = []

    with open(pdb) as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                record_counts[line[:6].strip()] += 1

                chain = line[21] if len(line) > 21 else "?"
                chain_counts[chain] += 1

                if len(atom_lines) < 5:
                    atom_lines.append(line.rstrip())

    print("\n==============================")
    print(name)
    print("File:", pdb)
    print("Record counts:", dict(record_counts))
    print("Chain characters:", dict(chain_counts))

    print("\nFirst 5 coordinate lines:")
    for line in atom_lines:
        print(repr(line))

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/7R73_HDOCK_top10")

for name in ["IgFold", "NBB2", "SimpleFold"]:
    pdb = next((base / name).rglob("model_1.pdb"))

    atom_lines = []

    with open(pdb) as f:
        for line in f:
            if line.startswith("ATOM"):
                atom_lines.append(line.rstrip())

    print("\n====================")
    print(name)
    print("Total atoms:", len(atom_lines))

    print("\nFirst molecule starts:")
    for line in atom_lines[:3]:
        print(line)

    print("\nSecond molecule starts:")
    for line in atom_lines[973:976]:
        print(line)

In [ ]:
from pathlib import Path

wanted = [
    "7R73_antigen_DOCK.pdb",
    "7R73A_IgFold_DOCK.pdb",
    "7R73A_NBB2_DOCK.pdb",
    "7R73A_SimpleFold_DOCK.pdb",
]

for name in wanted:
    matches = list(Path("/kaggle/input").rglob(name))

    print(f"\n{name}")
    for m in matches:
        print("✓", m)

In [ ]:
from pathlib import Path
import shutil

src = Path("/kaggle/input/datasets/mennanabiil/7r73docking")
dst = Path("/kaggle/working/HDOCK_REDO")
dst.mkdir(exist_ok=True)

files = {
    "RECEPTOR_gp120.pdb": src / "7R73_antigen_DOCK.pdb",
    "LIGAND_IgFold.pdb": src / "7R73A_IgFold_DOCK.pdb",
    "LIGAND_NBB2.pdb": src / "7R73A_NBB2_DOCK.pdb",
    "LIGAND_SimpleFold.pdb": src / "7R73A_SimpleFold_DOCK.pdb",
}

def atom_count(pdb):
    with open(pdb) as f:
        return sum(1 for line in f if line.startswith("ATOM"))

for new_name, source in files.items():
    target = dst / new_name
    shutil.copy(source, target)

    print(
        f"{new_name:25s}",
        "atoms =", atom_count(target),
        "✓"
    )

In [ ]:
from pathlib import Path
import tarfile

input_dir = Path("/kaggle/input")
out_dir = Path("/kaggle/working/7R73_DOCKQ")
out_dir.mkdir(exist_ok=True)

patterns = {
    "EXP": "top10_models-exp",
    "IgFold": "top10_models.igfree",
    "NBB2": "top10_models-nbb2free",
    "SimpleFold": "top10_models-simplefree",
}

for name, filename in patterns.items():

    matches = list(input_dir.rglob(filename))

    print(f"\n{name}")
    print("Archive:", matches)

    if not matches:
        continue

    archive = matches[0]
    target = out_dir / name
    target.mkdir(parents=True, exist_ok=True)

    with tarfile.open(archive, "r") as tar:
        tar.extractall(target)

    pdbs = list(target.rglob("model_*.pdb"))

    print("Extracted PDBs:", len(pdbs))
    for p in sorted(pdbs, key=lambda x: int(x.stem.split("_")[1])):
        print(" ", p)

In [ ]:
from pathlib import Path
from collections import Counter

base = Path("/kaggle/working/7R73_DOCKQ")

for name in ["EXP", "IgFold", "NBB2", "SimpleFold"]:

    pdbs = list((base / name).rglob("model_*.pdb"))
    pdbs = sorted(pdbs, key=lambda p: int(p.stem.split("_")[1]))

    print(f"\n{name}")

    for pdb in pdbs:
        chains = Counter()
        total_atoms = 0

        with open(pdb) as f:
            for line in f:
                if line.startswith("ATOM"):
                    total_atoms += 1
                    chains[line[21].strip()] += 1

        status = "✓" if total_atoms == 3256 else "⚠"

        print(
            f"{pdb.name:10s} "
            f"atoms={total_atoms:4d} "
            f"chains={dict(chains)} {status}"
        )

In [ ]:
!pip install -q DockQ==2.1.3

!DockQ --help | head -25

In [ ]:
from pathlib import Path
import subprocess

# Experimental HDOCK Top-1 pose
exp_model1 = next(
    Path("/kaggle/working/7R73_DOCKQ/EXP").rglob("model_1.pdb")
)

# Experimental/native crystal structure
native = Path(
    "/kaggle/input/datasets/mennanabiil/7r73docking/7R73.pdb"
)

print("Model :", exp_model1)
print("Native:", native)

cmd = [
    "DockQ",
    str(exp_model1),
    str(native),
    "--mapping", "GH:GA",
    "--short"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("\n--- DockQ OUTPUT ---")
print(result.stdout)

if result.stderr:
    print("\n--- STDERR ---")
    print(result.stderr)

In [ ]:
from pathlib import Path
import subprocess
import re

base = Path("/kaggle/working/7R73_DOCKQ")

native = Path(
    "/kaggle/input/datasets/mennanabiil/7r73docking/7R73.pdb"
)

groups = ["EXP", "IgFold", "NBB2", "SimpleFold"]

rows = []

pattern = re.compile(
    r"DockQ\s+([\d.]+)\s+"
    r"iRMSD\s+([\d.]+)\s+"
    r"LRMSD\s+([\d.]+)\s+"
    r"fnat\s+([\d.]+)\s+"
    r"fnonnat\s+([\d.]+)\s+"
    r"F1\s+([\d.]+)\s+"
    r"clashes\s+(\d+)"
)

def quality(score):
    if score < 0.23:
        return "Incorrect"
    elif score < 0.49:
        return "Acceptable"
    elif score < 0.80:
        return "Medium"
    else:
        return "High"


for group in groups:

    pdbs = list((base / group).rglob("model_*.pdb"))

    pdbs = sorted(
        pdbs,
        key=lambda p: int(p.stem.split("_")[1])
    )

    for pdb in pdbs:

        rank = int(pdb.stem.split("_")[1])

        result = subprocess.run(
            [
                "DockQ",
                str(pdb),
                str(native),
                "--mapping", "GH:GA",
                "--short"
            ],
            capture_output=True,
            text=True
        )

        match = pattern.search(result.stdout)

        if not match:
            print("FAILED:", group, pdb.name)
            print(result.stdout)
            print(result.stderr)
            continue

        dockq, irmsd, lrmsd, fnat, fnonnat, f1, clashes = match.groups()

        rows.append({
            "Model": group,
            "Rank": rank,
            "DockQ": float(dockq),
            "iRMSD": float(irmsd),
            "LRMSD": float(lrmsd),
            "fnat": float(fnat),
            "F1": float(f1),
            "Quality": quality(float(dockq))
        })


# Print all poses
print("\n================ ALL 40 POSES ================\n")

for r in rows:
    print(
        f"{r['Model']:10s} "
        f"Rank {r['Rank']:2d} | "
        f"DockQ {r['DockQ']:.3f} | "
        f"LRMSD {r['LRMSD']:.3f} | "
        f"fnat {r['fnat']:.3f} | "
        f"{r['Quality']}"
    )


# Summary
print("\n================ SUMMARY ================\n")

for group in groups:

    g = [r for r in rows if r["Model"] == group]

    top1 = next(r for r in g if r["Rank"] == 1)

    best = max(g, key=lambda r: r["DockQ"])

    print(group)
    print(
        f"  Top-1:        DockQ={top1['DockQ']:.3f}, "
        f"LRMSD={top1['LRMSD']:.3f}, "
        f"fnat={top1['fnat']:.3f}, "
        f"Quality={top1['Quality']}"
    )

    print(
        f"  Best Top-10:  Rank={best['Rank']}, "
        f"DockQ={best['DockQ']:.3f}, "
        f"LRMSD={best['LRMSD']:.3f}, "
        f"fnat={best['fnat']:.3f}, "
        f"Quality={best['Quality']}"
    )
    print()

In [ ]:
from pathlib import Path
import math

data = Path("/kaggle/input/datasets/mennanabiil/7r73docking")

antigen_native = data / "7R73_antigen_DOCK.pdb"
vhh_native     = data / "7R73_experimental_DOCK.pdb"

CUTOFF = 5.0


def read_atoms(pdb_path):
    atoms = []

    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue

            chain = line[21].strip()
            resname = line[17:20].strip()
            resnum = line[22:26].strip()
            icode = line[26].strip()

            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])

            atoms.append({
                "chain": chain,
                "residue": (resnum, icode, resname),
                "xyz": (x, y, z)
            })

    return atoms


def contact_residues(antigen_atoms, vhh_atoms, cutoff=5.0):
    cutoff2 = cutoff ** 2
    contacts = set()

    for a in antigen_atoms:
        ax, ay, az = a["xyz"]

        for b in vhh_atoms:
            bx, by, bz = b["xyz"]

            d2 = (
                (ax - bx)**2 +
                (ay - by)**2 +
                (az - bz)**2
            )

            if d2 <= cutoff2:
                contacts.add(a["residue"])
                break

    return contacts


antigen_atoms = read_atoms(antigen_native)
vhh_atoms = read_atoms(vhh_native)

native_epitope = contact_residues(
    antigen_atoms,
    vhh_atoms,
    cutoff=CUTOFF
)

native_epitope = sorted(
    native_epitope,
    key=lambda r: int(r[0])
)

print("Native gp120 epitope residues:", len(native_epitope))

for residue in native_epitope:
    print(residue)

In [ ]:
from pathlib import Path

base = Path("/kaggle/working/7R73_DOCKQ")
data = Path("/kaggle/input/datasets/mennanabiil/7r73docking")

CUTOFF = 5.0


# ---------- Read PDB atoms ----------
def read_chain_atoms(pdb_path, chain_id=None):
    atoms = []

    with open(pdb_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue

            chain = line[21].strip()

            if chain_id is not None and chain != chain_id:
                continue

            resname = line[17:20].strip()
            resnum = line[22:26].strip()
            icode = line[26].strip()

            x = float(line[30:38])
            y = float(line[38:46])
            z = float(line[46:54])

            atoms.append({
                "chain": chain,
                "residue": (resnum, icode, resname),
                "xyz": (x, y, z)
            })

    return atoms


# ---------- Find antigen residues contacting VHH ----------
def contact_residues(antigen_atoms, vhh_atoms, cutoff=5.0):
    cutoff2 = cutoff ** 2
    contacts = set()

    for a in antigen_atoms:
        ax, ay, az = a["xyz"]

        for b in vhh_atoms:
            bx, by, bz = b["xyz"]

            d2 = (
                (ax - bx)**2 +
                (ay - by)**2 +
                (az - bz)**2
            )

            if d2 <= cutoff2:
                contacts.add(a["residue"])
                break

    return contacts


# ---------- Native experimental epitope ----------
native_antigen = read_chain_atoms(
    data / "7R73_antigen_DOCK.pdb",
    "G"
)

native_vhh = read_chain_atoms(
    data / "7R73_experimental_DOCK.pdb",
    "H"
)

native_epitope = contact_residues(
    native_antigen,
    native_vhh,
    CUTOFF
)


# ---------- DockQ-selected ranks ----------
best_ranks = {
    "EXP": 1,
    "IgFold": 5,
    "NBB2": 1,
    "SimpleFold": 9
}


def get_model_pdb(group, rank):
    matches = list(
        (base / group).rglob(f"model_{rank}.pdb")
    )

    if len(matches) != 1:
        raise RuntimeError(
            f"{group} rank {rank}: found {len(matches)} files"
        )

    return matches[0]


def evaluate_epitope(pdb):
    antigen = read_chain_atoms(pdb, "G")
    vhh = read_chain_atoms(pdb, "H")

    predicted = contact_residues(
        antigen,
        vhh,
        CUTOFF
    )

    intersection = native_epitope & predicted
    union = native_epitope | predicted

    jaccard = len(intersection) / len(union) if union else 0
    recall = len(intersection) / len(native_epitope)
    precision = len(intersection) / len(predicted) if predicted else 0

    return {
        "predicted": len(predicted),
        "correct": len(intersection),
        "jaccard": jaccard,
        "recall": recall,
        "precision": precision
    }


print("Native epitope:", len(native_epitope), "residues")
print("\n================ EPITOPE RECOVERY ================\n")


for group in ["EXP", "IgFold", "NBB2", "SimpleFold"]:

    # Top-1
    top1_pdb = get_model_pdb(group, 1)
    top1 = evaluate_epitope(top1_pdb)

    # Best DockQ among Top-10
    best_rank = best_ranks[group]
    best_pdb = get_model_pdb(group, best_rank)
    best = evaluate_epitope(best_pdb)

    print(group)

    print(
        f"  Top-1 (Rank 1): "
        f"Predicted={top1['predicted']}, "
        f"Correct={top1['correct']}/26, "
        f"Jaccard={top1['jaccard']:.3f}, "
        f"Recall={top1['recall']:.3f}, "
        f"Precision={top1['precision']:.3f}"
    )

    print(
        f"  Best DockQ (Rank {best_rank}): "
        f"Predicted={best['predicted']}, "
        f"Correct={best['correct']}/26, "
        f"Jaccard={best['jaccard']:.3f}, "
        f"Recall={best['recall']:.3f}, "
        f"Precision={best['precision']:.3f}"
    )

    print()